[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/cdg-ime/blob/main/PreProcessamento.ipynb)

# Ciência de Dados Geoespaciais - Pré-Processamento

**Maj Diego - 2° Semestre / 2026**

**Objetivos**

1. Identificar problemas nos dados;
2. Aplicar métodos de seleção de dados;
3. Aplicar métodos de limpeza de dados;
4. Aplicar métodos de completamento de dados.

**Referência:**
- GARCIA, S. et al. *Data Preprocessing in Data Mining*. Springer, 2015.
- REY, S.J.; ARRIBAS-BEL, D.; WOLF, L.J. *Geographic Data Science with Python*. CRC Press, 2023.
- MCKINNEY, W. *Python for Data Analysis*. O'Reilly, 2022.

## O Contexto

Após a **prospecção**, raramente os dados chegam prontos para análise. Na prática, estima-se que **60–80% do tempo** de um cientista de dados é gasto nesta etapa.

```
┌─────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  Prospecção │───▶│ Pré-Processo │───▶│  Exploração  │───▶│  Modelagem   │───▶│  Comunicação │
└─────────────┘    └──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
                       ← você está aqui
```

### Dataset de referência desta aula

Usaremos um dataset sintético de **postos pluviométricos** do estado do Rio de Janeiro, propositalmente com problemas comuns do mundo real.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt

# ── Dataset sintético: postos pluviométricos do RJ ───────────
dados_brutos = {
    'id'           : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    'nome'         : ['Posto A', 'Posto B', 'Posto C', 'Posto D', 'posto a',
                       'Posto F', 'Posto G', 'Posto H', 'Posto I', 'Posto J', 'Posto K'],
    'municipio'    : ['Niterói', 'Rio de Janeiro', 'Petrópolis', 'Angra dos Reis',
                       'Niterói', 'Volta Redonda', None, 'Campos', 'Resende', 'Macaé', 'Cabo Frio'],
    'latitude'     : [-22.88, -22.90, -22.51, -23.01, -22.88,
                       -22.52, -22.75, -21.75, -22.47, -22.38, -22.88],
    'longitude'    : [-43.10, -43.17, -43.18, -44.32, -43.10,
                       -44.08, -250.0, -41.33, -44.45, -41.79, -42.02],   # Posto G: lon inválida!
    'precip_jan_mm': [120.5, 135.0, 98.3,  None,  120.5,
                       88.7,  77.2,  200.0,  61.4,  110.2, 95.8],
    'precip_fev_mm': [110.2, 142.1, 91.7,  88.0,  110.2,
                       80.1,  70.5,  9999.0, 58.2,  105.3, 91.1],          # 9999 = valor sentinela
    'altitude_m'   : [15,    10,    843,   5,     15,
                       390,   210,   14,    430,   12,    5],
    'ativo'        : [True, True, True, True, True, True, True, False, True, True, True]
}

df = pd.DataFrame(dados_brutos)
print(f"Shape: {df.shape}")
df

## 1. Identificar problemas nos dados

O primeiro passo é fazer um **diagnóstico completo** antes de qualquer intervenção. Tratar dados sem diagnóstico é como prescrever remédio sem examinar o paciente.

### 1.1 Taxonomia dos problemas

| Categoria | Problema | Exemplo no dataset |
|-----------|----------|--------------------|
| **Completude** | Valores ausentes (`NaN`) | `municipio` do Posto G, `precip_jan_mm` do Posto D |
| **Consistência** | Duplicatas | Posto A e Posto a (mesmo local) |
| **Validade** | Valores fora do domínio | `longitude = -250.0` (impossível) |
| **Validade** | Valores sentinela | `precip_fev_mm = 9999` (código de erro legado) |
| **Relevância** | Registros inativos | `ativo = False` no Posto H |
| **Padronização** | Inconsistência de grafia | `'posto a'` vs `'Posto A'` |

### 1.2 Ferramentas de diagnóstico

In [ ]:
# ── 1. Tipos de dados e visão geral ──────────────────────────
print("=" * 50)
print("TIPOS DE DADOS")
print("=" * 50)
print(df.dtypes)

print("\n" + "=" * 50)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 50)
df.describe()

In [ ]:
# ── 2. Auditoria de valores ausentes ─────────────────────────
ausentes = df.isnull().sum()
pct = (ausentes / len(df) * 100).round(1)

auditoria = pd.DataFrame({
    'Nulos'     : ausentes,
    'Percentual': pct
}).query('Nulos > 0')

print("Colunas com valores ausentes:")
print(auditoria)

In [ ]:
# ── 3. Verificação de domínio (regras de negócio) ─────────────
print("Coordenadas fora do domínio válido (lat: -90/90, lon: -180/180):")
invalidos_coord = df[
    (df['latitude'].abs()  > 90)  |
    (df['longitude'].abs() > 180)
]
print(invalidos_coord[['id','nome','latitude','longitude']])

print("\nValores sentinela em precipitação (>= 9000):")
print(df[df[['precip_jan_mm','precip_fev_mm']].ge(9000).any(axis=1)]
      [['id','nome','precip_jan_mm','precip_fev_mm']])

print("\nDuplicatas por coordenada:")
print(df[df.duplicated(subset=['latitude','longitude'], keep=False)]
      [['id','nome','latitude','longitude']])

In [ ]:
# ── 4. Heatmap visual de ausentes ────────────────────────────
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 4))
mascara = df.isnull()
ax.imshow(mascara.values.T, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=1)
ax.set_xticks(range(len(df)))
ax.set_xticklabels([f"#{i}" for i in df['id']], fontsize=8)
ax.set_yticks(range(len(df.columns)))
ax.set_yticklabels(df.columns, fontsize=9)
ax.set_title("Mapa de valores ausentes (vermelho = NaN)")
plt.tight_layout()
plt.show()

## 2. Aplicar métodos de seleção de dados

Selecionar significa **reduzir o escopo** do dataset mantendo apenas o que é relevante para a análise. Há duas dimensões: **linhas** (registros) e **colunas** (atributos).

### 2.1 Seleção de registros (linhas)

| Critério | Quando usar |
|----------|-------------|
| Filtro por atributo | Remover registros inativos, fora da área de estudo |
| Filtro espacial | Selecionar feições dentro de um polígono ou buffer |
| Amostragem | Dataset muito grande; análise exploratória inicial |

In [ ]:
# ── Seleção por atributo: apenas postos ativos ────────────────
df_ativos = df[df['ativo'] == True].copy()
print(f"Antes: {len(df)} registros | Após filtro 'ativo': {len(df_ativos)} registros")

# ── Seleção por domínio: coordenadas válidas ──────────────────
df_validos = df_ativos[
    (df_ativos['latitude'].abs()  <= 90) &
    (df_ativos['longitude'].abs() <= 180)
].copy()
print(f"Após filtro de coordenadas válidas: {len(df_validos)} registros")
df_validos[['id','nome','latitude','longitude']]

In [ ]:
# ── Seleção espacial: postos dentro de um raio de 150 km do Rio de Janeiro
from shapely.geometry import Point

gdf = gpd.GeoDataFrame(
    df_validos,
    geometry=[Point(lon, lat) for lon, lat in
              zip(df_validos['longitude'], df_validos['latitude'])],
    crs='EPSG:4326'
).to_crs('EPSG:32723')  # Projeção métrica para medir distâncias

# Centro do Rio de Janeiro
centro_rj = gpd.GeoDataFrame(
    geometry=[Point(-43.17, -22.90)], crs='EPSG:4326'
).to_crs('EPSG:32723').geometry[0]

raio_m = 150_000  # 150 km
gdf_proximo = gdf[gdf.geometry.distance(centro_rj) <= raio_m].copy()

print(f"Postos a até 150 km do Rio de Janeiro: {len(gdf_proximo)}")
print(gdf_proximo[['id','nome','municipio']].to_string(index=False))

### 2.2 Seleção de atributos (colunas)

Manter apenas colunas relevantes reduz o ruído e melhora o desempenho dos modelos subsequentes.

In [ ]:
# ── Seleção manual de colunas de interesse ────────────────────
colunas_interesse = ['id', 'nome', 'municipio', 'latitude', 'longitude',
                     'precip_jan_mm', 'precip_fev_mm', 'altitude_m']

df_sel = df_validos[colunas_interesse].copy()

# ── Verificar correlação entre colunas numéricas ──────────────
# Colunas com correlação muito alta podem ser redundantes
print("Correlação entre variáveis numéricas:")
print(df_sel.select_dtypes('number').corr().round(2))

## 3. Aplicar métodos de limpeza de dados

Limpeza trata os problemas **identificados no diagnóstico**. Cada tipo de problema tem seu conjunto de tratamentos adequados.

### 3.1 Remoção e tratamento de duplicatas

In [ ]:
# ── Detectar duplicatas por coordenada ────────────────────────
print("Registros duplicados (mesma coordenada):")
mask_dup = df_sel.duplicated(subset=['latitude', 'longitude'], keep=False)
print(df_sel[mask_dup][['id','nome','latitude','longitude']])

# ── Normalizar o nome antes de comparar ───────────────────────
df_sel['nome_norm'] = df_sel['nome'].str.strip().str.title()

# Remover duplicata, mantendo o primeiro registro
df_sem_dup = df_sel.drop_duplicates(subset=['latitude', 'longitude'], keep='first').copy()
df_sem_dup = df_sem_dup.drop(columns=['nome_norm'])

print(f"\nApós remoção de duplicatas: {len(df_sem_dup)} registros")

### 3.2 Padronização de texto e categorias

In [ ]:
# ── Padronização de strings ───────────────────────────────────
df_limpo = df_sem_dup.copy()

# Nome: Title Case + sem espaços extras
df_limpo['nome']       = df_limpo['nome'].str.strip().str.title()

# Município: strip + Title Case
df_limpo['municipio']  = df_limpo['municipio'].str.strip().str.title()

print("Nomes após padronização:")
print(df_limpo[['id','nome','municipio']])

### 3.3 Tratamento de valores sentinela e fora de domínio

In [ ]:
# ── Substituir sentinelas por NaN ────────────────────────────
# Valores >= 9000 são códigos de erro legados do sistema de coleta
SENTINEL = 9000
cols_precip = ['precip_jan_mm', 'precip_fev_mm']

for col in cols_precip:
    n_sentinelas = (df_limpo[col] >= SENTINEL).sum()
    if n_sentinelas > 0:
        print(f"  {col}: {n_sentinelas} valor(es) sentinela substituído(s) por NaN")
    df_limpo[col] = df_limpo[col].where(df_limpo[col] < SENTINEL, other=np.nan)

# ── Verificar precipitações negativas (fisicamente impossível)
for col in cols_precip:
    n_neg = (df_limpo[col] < 0).sum()
    if n_neg > 0:
        print(f"  {col}: {n_neg} valor(es) negativo(s) → convertidos para NaN")
        df_limpo[col] = df_limpo[col].where(df_limpo[col] >= 0, other=np.nan)

print("\nDataset após tratamento de sentinelas:")
print(df_limpo[['id','nome'] + cols_precip])

### 3.4 Detecção e tratamento de outliers

**Outliers** são valores que se afastam significativamente dos demais. Nem todo outlier é erro — pode ser um evento extremo real. A decisão de remover ou manter deve ser embasada no contexto.

Método clássico: **IQR (Intervalo Interquartil)**

```
Limite inferior = Q1 - 1.5 × IQR
Limite superior = Q3 + 1.5 × IQR
```

In [ ]:
def detectar_outliers_iqr(serie):
    """Retorna máscara booleana com True onde há outlier pelo critério IQR."""
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    return (serie < Q1 - 1.5 * IQR) | (serie > Q3 + 1.5 * IQR)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col in zip(axes, cols_precip):
    serie = df_limpo[col].dropna()
    outliers = detectar_outliers_iqr(serie)
    ax.boxplot(serie, vert=True, patch_artist=True,
               boxprops=dict(facecolor='#AED6F1'))
    ax.scatter(
        [1] * outliers.sum(),
        serie[outliers],
        color='red', zorder=5, label=f"{outliers.sum()} outlier(s)"
    )
    ax.set_title(col)
    ax.legend()

plt.suptitle("Boxplot com outliers destacados (IQR × 1.5)", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Aplicar métodos de completamento de dados

**Completamento** (ou *imputation*) é o processo de preencher valores ausentes com estimativas razoáveis. Há uma hierarquia de estratégias — da mais simples à mais sofisticada.

### 4.1 Estratégias de completamento

| Estratégia | Descrição | Quando usar |
|-----------|-----------|-------------|
| **Remoção** | Descarta linhas/colunas com NaN | % ausente alta, sem padrão espacial |
| **Média / Mediana** | Substitui pelo valor central da coluna | Distribuições simétricas / com outliers |
| **Moda** | Substitui pelo valor mais frequente | Variáveis categóricas |
| **Interpolação temporal** | Usa vizinhos no tempo | Séries temporais |
| **Interpolação espacial** | Usa vizinhos no espaço | Dados geoespaciais |
| **Modelo preditivo** | Treina um modelo para prever | Dados complexos com padrão |

### 4.2 Remoção de registros irrecuperáveis

In [ ]:
print("NaNs antes do completamento:")
print(df_limpo.isnull().sum())

# Registros sem município E sem coordenadas válidas são irrecuperáveis
df_comp = df_limpo.dropna(subset=['latitude', 'longitude']).copy()
print(f"\nApós remover registros sem coordenadas: {len(df_comp)} registros")

### 4.3 Completamento de atributos categóricos (município)

In [ ]:
# ── Municipio ausente: buscar via geocodificação reversa ──────
# Estratégia simples: usar a cidade mais próxima conhecida
# (em produção: usar uma API de geocodificação reversa)

print("Postos com município ausente:")
print(df_comp[df_comp['municipio'].isna()][['id','nome','latitude','longitude']])

# Simulando o resultado de uma geocodificação reversa
# (ex.: Nominatim / IBGE API de malhas)
municipios_recuperados = {
    # id: municipio inferido pelas coordenadas
    # (neste dataset sintético, o Posto G foi removido por coordenada inválida)
}

# Para campos categóricos sem regra clara, podemos usar 'Desconhecido'
df_comp['municipio'] = df_comp['municipio'].fillna('Desconhecido')
print("\nApós completamento de município:")
print(df_comp[['id','nome','municipio']])

### 4.4 Completamento de atributos numéricos (precipitação)

Para variáveis contínuas com distribuição razoável, a **mediana** é mais robusta que a média na presença de outliers.

In [ ]:
# ── Completamento por mediana da coluna ───────────────────────
for col in cols_precip:
    mediana = df_comp[col].median()
    n_antes = df_comp[col].isna().sum()
    df_comp[col] = df_comp[col].fillna(mediana)
    print(f"{col}: {n_antes} NaN(s) preenchido(s) com mediana = {mediana:.1f} mm")

print("\nNaNs após completamento numérico:")
print(df_comp[cols_precip].isnull().sum())

### 4.5 Completamento por interpolação espacial

Em dados geoespaciais, o princípio de **Tobler** — *"tudo está relacionado com tudo, mas coisas próximas mais do que coisas distantes"* — sugere que usar vizinhos espaciais é mais adequado que uma estatística global.

O método mais simples é o **Inverso da Distância Ponderada (IDW)**:

$$\hat{z}(p) = \frac{\sum_{i=1}^{n} \frac{z_i}{d_i^\beta}}{\sum_{i=1}^{n} \frac{1}{d_i^\beta}}$$

onde $d_i$ é a distância ao ponto $i$, e $\beta$ é o expoente de ponderação (tipicamente 2).

In [ ]:
from scipy.spatial import cKDTree

def idw_completar(gdf, col, beta=2, n_vizinhos=3):
    """
    Completa NaNs de `col` usando IDW com os `n_vizinhos` mais próximos.
    `gdf` deve estar em projeção métrica (ex.: EPSG:32723).
    """
    gdf = gdf.copy()
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))

    mask_nan  = gdf[col].isna()
    mask_val  = ~mask_nan

    if mask_nan.sum() == 0:
        return gdf  # Nada a fazer

    # KD-tree apenas com pontos que têm valor
    tree = cKDTree(coords[mask_val])
    valores = gdf.loc[mask_val, col].values

    # Para cada ponto com NaN, buscar os n_vizinhos mais próximos
    dists, idxs = tree.query(coords[mask_nan], k=min(n_vizinhos, mask_val.sum()))

    # IDW
    pesos = 1.0 / (dists ** beta + 1e-10)  # Evita divisão por zero
    estimativas = np.sum(pesos * valores[idxs], axis=1) / np.sum(pesos, axis=1)

    gdf.loc[mask_nan, col] = estimativas.round(1)
    return gdf


# Criar GeoDataFrame para usar IDW
gdf_final = gpd.GeoDataFrame(
    df_comp,
    geometry=[Point(lon, lat) for lon, lat in
              zip(df_comp['longitude'], df_comp['latitude'])],
    crs='EPSG:4326'
).to_crs('EPSG:32723')

# Introduzindo um NaN artificial para demonstrar IDW
gdf_final.loc[gdf_final['id'] == 1, 'precip_jan_mm'] = np.nan
print("NaN introduzido no Posto 1 (Niterói):")
print(gdf_final[gdf_final['id']==1][['id','nome','precip_jan_mm']])

gdf_final = idw_completar(gdf_final, 'precip_jan_mm', beta=2, n_vizinhos=3)
print("\nApós IDW:")
print(gdf_final[gdf_final['id']==1][['id','nome','precip_jan_mm']])

### 4.6 Dataset final — visão consolidada

In [ ]:
gdf_plot = gdf_final.to_crs('EPSG:4326')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, titulo, cor in zip(
    axes,
    ['precip_jan_mm', 'precip_fev_mm'],
    ['Precipitação - Janeiro (mm)', 'Precipitação - Fevereiro (mm)'],
    ['Blues', 'Purples']
):
    gdf_plot.plot(column=col, ax=ax, legend=True,
                  cmap=cor, markersize=100, edgecolor='black',
                  legend_kwds={'label': 'mm', 'shrink': 0.7})
    for _, row in gdf_plot.iterrows():
        ax.annotate(row['nome'], xy=(row.geometry.x, row.geometry.y),
                    xytext=(3, 5), textcoords='offset points', fontsize=7)
    ax.set_title(titulo)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle("Postos Pluviométricos — Dados Pré-Processados", fontsize=13)
plt.tight_layout()
plt.show()

print("\nDataset final:")
gdf_plot[['id','nome','municipio','precip_jan_mm','precip_fev_mm','altitude_m']]

## Complemento

### Resumo do pipeline de pré-processamento

```
Dados brutos
     │
     ▼
[1. DIAGNÓSTICO] ── isnull(), describe(), boxplot, mapa de ausentes
     │
     ▼
[2. SELEÇÃO] ─────── filtro por atributo, filtro espacial, seleção de colunas
     │
     ▼
[3. LIMPEZA] ─────── duplicatas, padronização, sentinelas, outliers
     │
     ▼
[4. COMPLETAMENTO] ─ remoção, mediana, interpolação espacial (IDW)
     │
     ▼
Dados prontos para Exploração
```

### Checklist de qualidade antes de avançar

- [ ] Nenhuma coluna essencial com NaN
- [ ] Coordenadas dentro dos limites válidos (`lat ∈ [-90, 90]`, `lon ∈ [-180, 180]`)
- [ ] CRS definido e consistente
- [ ] Sem duplicatas por identificador único
- [ ] Textos padronizados (sem variações de capitalização)
- [ ] Valores sentinela substituídos
- [ ] Outliers documentados (removidos ou justificados)
- [ ] Todas as decisões de tratamento registradas (rastreabilidade)

### Bibliotecas úteis

```python
# ydata-profiling: relatório automático de qualidade de dados
from ydata_profiling import ProfileReport
report = ProfileReport(df, title="Relatório de Qualidade")
report.to_notebook_iframe()

# missingno: visualização avançada de padrões de ausência
import missingno as msno
msno.matrix(df)
msno.heatmap(df)   # Correlação entre ausências

# pyproj / geopandas: reprojeção e validação de geometrias
gdf['valida'] = gdf.geometry.is_valid
gdf_corrigido = gdf.copy()
gdf_corrigido.geometry = gdf.geometry.buffer(0)  # Corrige geometrias inválidas
```

## Lista de exercícios complementares

**Exercício 1.** Carregue um CSV de sua escolha (sugestão: focos de queimadas do INPE). Use `describe()`, `isnull().sum()` e um boxplot para fazer o diagnóstico completo. Quantos e quais problemas você identificou?

**Exercício 2.** Dado o dataset de postos pluviométricos desta aula, adicione manualmente 3 novos problemas (um valor impossível, uma duplicata e um sentinela diferente). Aplique o pipeline completo de limpeza.

**Exercício 3.** Usando a malha municipal do IBGE e `geopandas`, selecione espacialmente todos os postos pluviométricos que caem dentro dos limites do estado do Rio de Janeiro. Quantos são descartados por cair fora?

**Exercício 4.** Compare três estratégias de completamento para a coluna `precip_jan_mm`: (a) média global, (b) mediana global, (c) IDW com 3 vizinhos. Para cada estratégia, calcule o erro absoluto médio em relação ao valor original que você ocultou.

**Exercício 5.** Pesquise e aplique a biblioteca `missingno` para visualizar os padrões de ausência do dataset. Os valores faltam de forma aleatória (*MCAR*) ou há um padrão espacial (*MNAR*)?

**Exercício 6 (desafio).** Monte um pipeline completo de pré-processamento usando `pandas` e `geopandas` encapsulado em uma classe Python `PreProcessadorGeoespacial`, com métodos: `.diagnosticar()`, `.selecionar()`, `.limpar()` e `.completar()`. O método `.diagnosticar()` deve retornar um relatório em formato de dicionário.